# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each record set, field, and column can be referenced by its `@id`. We enumerate the available record sets and their fields below.

In [ ]:
# List available record sets and their fields
record_sets = metadata.record_sets

if not record_sets:
    print('No record sets found in metadata. Please check the dataset schema or contact the dataset provider.')
else:
    for rs in record_sets:
        print(f"RecordSet ID: {rs['@id']}")
        print(f"  Name: {rs.get('name', '(no name)')}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print(f"  Fields:")
        for f in fields:
            if isinstance(f, dict):
                fid = f.get('@id', str(f))
                print(f"    Field ID: {fid}")
            else:
                print(f"    Field ID: {f}")
        print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For this example, we'll load the first available record set (if present).

In [ ]:
dataframes = {}

# Check for available record sets
if not metadata.record_sets:
    print('No record sets found for extraction.')
else:
    # For demonstration, use the first available record set
    main_rs = metadata.record_sets[0]
    main_rs_id = main_rs['@id']
    print(f"Using record set: {main_rs_id}")

    # Load all records from this record set
    records = list(dataset.records(record_set=main_rs_id))
    if len(records) == 0:
        print(f"No records found in record set {main_rs_id}")
    else:
        df = pd.DataFrame(records)
        dataframes[main_rs_id] = df
        print(f"Loaded columns: {df.columns.tolist()}")
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis. Modify the field selections to match what's available in your data.

In [ ]:
# Example EDA using protein quantification columns (modify as needed based on real columns).
import numpy as np

if not dataframes:
    print('No DataFrame loaded for EDA.')
else:
    df = list(dataframes.values())[0]
    rsid = list(dataframes.keys())[0]
    # Try to find a numeric field - guess from typical protein mass spec column names
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int] or pd.api.types.is_numeric_dtype(df[col])]
    if len(numeric_candidates) == 0:
        # Otherwise, try to infer numeric fields from their names
        for col in df.columns:
            if 'abundance' in col.lower() or 'mw' in col.lower() or 'score' in col.lower() or 'percent' in col.lower():
                numeric_candidates.append(col)
    if not numeric_candidates:
        print('No obvious numeric field found. Please specify one.')
    else:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")

        # Filtering: Remove records with missing or zero value
        filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce').fillna(0) > 0]

        # Optionally filter using a threshold
        threshold = filtered_df[numeric_field].mean()
        filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (first 5 records):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field
        group_field_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
        group_field = group_field_candidates[0] if group_field_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
        else:
            print('No suitable group field found.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: Histogram and boxplot for the selected numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'filtered_df' in locals():
    plt.figure(figsize=(12,4))
    plt.subplot(1, 2, 1)
    sns.histplot(filtered_df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f'Histogram of {numeric_field}')

    plt.subplot(1, 2, 2)
    sns.boxplot(x=filtered_df[numeric_field].dropna())
    plt.title(f'Boxplot of {numeric_field}')
    plt.tight_layout()
    plt.show()
    
    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No data to plot.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains abundant protein mass spectrometry features, with field and record set structures accessible via their `@id`.
- Basic filtering, normalization, and grouping can be performed with `mlcroissant` and pandas utilizing Croissant schema identifiers.
- Visualizations reveal the shape and group-wise distribution of selected numeric variables.
- For custom or deeper analyses, consult the Croissant field definitions and documentation for detailed semantic meaning and identifier mapping.